In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="10-personalized-news-feed",
    notebook_name="02_ranking_and_personalization.ipynb"
)

# Ranking and Personalization: Deep Dive

## The Big Idea (For a 12-Year-Old)

In the last notebook, we built the overall system. Now we are going to zoom in on the **brain** of the news feed -- the ranking model. This is like looking under the hood of a car to understand exactly how the engine works.

The key question: **How does the model figure out which posts YOU will love?**

It turns out the answer mostly comes down to three things:
1. **Who posted it** -- Do you usually like stuff from this person? (User-author affinity)
2. **What is in the post** -- Is it about sports? Does it have a funny picture? (Content understanding)
3. **What your friends think** -- Did your best friend already like it? (Social graph features)

---

## Staff-Level Technical Summary

This notebook covers:
- **User-author affinity scoring** with interaction history and decay functions
- **Content understanding** via NLP (BERT embeddings) and CV (image embeddings)
- **Social graph features** (friends who engaged, social proof signals)
- **Embedding-based features** for users, posts, and cross-features
- **Multi-task DNN architecture** with shared-bottom and expert layers
- **Negative feedback handling** (hide, unfollow, report) and its asymmetric impact on ranking

## 1. User-Author Affinity: The Most Important Feature

According to Facebook's published research, **the relationship between a user and the post's author is the single strongest predictor of engagement**. This makes intuitive sense -- you are far more likely to like a post from your best friend than from a random page you followed two years ago.

Think of it like school: if your best friend tells you a joke, you will probably laugh. If a stranger in the hallway tells the same joke, you might just walk by. The joke is the same -- what changed is **who told it**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# === Generating Synthetic Interaction History ===

np.random.seed(42)

# Create a realistic interaction history between users and authors
n_users = 10
n_authors = 15
n_interactions = 5000

# Some user-author pairs have strong affinity, others weak
# This simulates real social dynamics
affinity_matrix = np.random.beta(0.5, 2.0, (n_users, n_authors))  # Most pairs have LOW affinity
# Make a few pairs have HIGH affinity (close friends)
for _ in range(15):
    u, a = np.random.randint(0, n_users), np.random.randint(0, n_authors)
    affinity_matrix[u, a] = np.random.uniform(0.6, 0.95)

# Generate interaction log
interactions = []
base_time = datetime(2024, 1, 1)

for _ in range(n_interactions):
    user_id = np.random.randint(0, n_users)
    author_id = np.random.randint(0, n_authors)
    affinity = affinity_matrix[user_id, author_id]
    
    # Higher affinity -> more likely to engage
    timestamp = base_time + timedelta(days=np.random.uniform(0, 90))
    
    # Every interaction starts as an impression
    interactions.append({
        'user_id': user_id,
        'author_id': author_id,
        'type': 'impression',
        'timestamp': timestamp
    })
    
    # Generate reactions based on affinity
    if np.random.random() < affinity * 0.5:  # click
        interactions.append({'user_id': user_id, 'author_id': author_id, 'type': 'click', 'timestamp': timestamp})
    if np.random.random() < affinity * 0.3:  # like
        interactions.append({'user_id': user_id, 'author_id': author_id, 'type': 'like', 'timestamp': timestamp})
    if np.random.random() < affinity * 0.1:  # comment
        interactions.append({'user_id': user_id, 'author_id': author_id, 'type': 'comment', 'timestamp': timestamp})
    if np.random.random() < affinity * 0.05:  # share
        interactions.append({'user_id': user_id, 'author_id': author_id, 'type': 'share', 'timestamp': timestamp})
    if np.random.random() < (1 - affinity) * 0.05:  # hide (anti-correlated with affinity)
        interactions.append({'user_id': user_id, 'author_id': author_id, 'type': 'hide', 'timestamp': timestamp})

interactions_df = pd.DataFrame(interactions)
interactions_df = interactions_df.sort_values('timestamp').reset_index(drop=True)

print(f"Generated {len(interactions_df)} interaction records")
print(f"\nInteraction type distribution:")
for itype, count in interactions_df['type'].value_counts().items():
    print(f"  {itype:<12}: {count:>5} ({count/len(interactions_df)*100:.1f}%)")

In [ ]:
# === Computing User-Author Affinity Scores ===

def compute_affinity_score(user_id, author_id, interactions_df, current_time=None,
                           decay_half_life_days=30):
    """
    Compute the affinity score between a user and an author.
    
    12-year-old version:
    We look at everything you've done with this person's posts:
    - Did you like their posts? (good sign)
    - Did you comment? (even better -- you took time to write!)
    - Did you hide their posts? (bad sign)
    And we care MORE about recent interactions (yesterday matters more than last month).
    
    Staff-level detail:
    - Computes per-reaction-type rates (likes/impressions, comments/impressions, etc.)
    - Applies exponential time decay (recent interactions weighted higher)
    - Combines into a single affinity score with configurable reaction weights
    """
    if current_time is None:
        current_time = datetime(2024, 4, 1)
    
    # Filter to this user-author pair
    pair = interactions_df[
        (interactions_df['user_id'] == user_id) &
        (interactions_df['author_id'] == author_id)
    ].copy()
    
    if len(pair) == 0:
        return 0.0, {}  # No history -> cold start
    
    # Apply time decay: recent interactions matter more
    days_ago = (current_time - pair['timestamp']).dt.total_seconds() / 86400
    decay_weights = np.exp(-0.693 * days_ago / decay_half_life_days)  # 0.693 = ln(2)
    pair['decay_weight'] = decay_weights.values
    
    # Compute weighted rates per reaction type
    impressions = pair[pair['type'] == 'impression']
    total_weighted_impressions = impressions['decay_weight'].sum()
    if total_weighted_impressions == 0:
        return 0.0, {}
    
    reaction_weights = {
        'click': 1.0,
        'like': 3.0,
        'comment': 5.0,
        'share': 8.0,
        'hide': -10.0,
    }
    
    details = {}
    total_score = 0.0
    
    for reaction, weight in reaction_weights.items():
        reaction_data = pair[pair['type'] == reaction]
        weighted_count = reaction_data['decay_weight'].sum() if len(reaction_data) > 0 else 0
        rate = weighted_count / total_weighted_impressions
        contribution = rate * weight
        total_score += contribution
        details[reaction] = {'rate': rate, 'weight': weight, 'contribution': contribution}
    
    # Normalize to [0, 1] using sigmoid
    normalized_score = 1.0 / (1.0 + np.exp(-total_score))
    
    return normalized_score, details


# Demo: compute affinity for a few user-author pairs
print("=== User-Author Affinity Scores ===")
print(f"{'User':<8} {'Author':<8} {'Affinity':<12} {'True Affinity':<15} {'Details'}")
print("-" * 80)

test_pairs = [(0, 0), (0, 3), (2, 5), (5, 10), (7, 2)]
for user_id, author_id in test_pairs:
    score, details = compute_affinity_score(user_id, author_id, interactions_df)
    true_aff = affinity_matrix[user_id, author_id]
    detail_str = ", ".join([f"{k}: {v['rate']:.2f}" for k, v in details.items() if v['rate'] > 0.01])
    print(f"User_{user_id:<4} Auth_{author_id:<4} {score:<12.4f} {true_aff:<15.4f} {detail_str}")

In [ ]:
# === Visualizing Time Decay ===

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Decay curves with different half-lives
days = np.linspace(0, 90, 100)
half_lives = [7, 14, 30, 60]
colors = ['#e74c3c', '#e67e22', '#3498db', '#2ecc71']

for hl, color in zip(half_lives, colors):
    weights = np.exp(-0.693 * days / hl)
    axes[0].plot(days, weights, color=color, linewidth=2, label=f'Half-life = {hl} days')

axes[0].set_xlabel('Days Ago', fontsize=12)
axes[0].set_ylabel('Weight', fontsize=12)
axes[0].set_title('Time Decay: How Much Do Old Interactions Matter?', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)

# Plot 2: Affinity matrix heatmap
# Compute affinity scores for a subset
affinity_scores = np.zeros((5, 5))
for u in range(5):
    for a in range(5):
        score, _ = compute_affinity_score(u, a, interactions_df)
        affinity_scores[u, a] = score

im = axes[1].imshow(affinity_scores, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
axes[1].set_xlabel('Author ID', fontsize=12)
axes[1].set_ylabel('User ID', fontsize=12)
axes[1].set_title('User-Author Affinity Heatmap', fontsize=13, fontweight='bold')
axes[1].set_xticks(range(5))
axes[1].set_yticks(range(5))

# Add text annotations
for u in range(5):
    for a in range(5):
        axes[1].text(a, u, f'{affinity_scores[u, a]:.2f}',
                     ha='center', va='center', fontsize=10,
                     color='white' if affinity_scores[u, a] > 0.6 or affinity_scores[u, a] < 0.3 else 'black')

plt.colorbar(im, ax=axes[1], label='Affinity Score')
plt.tight_layout()
plt.show()

print("Left: Time decay curves -- with a 30-day half-life, an interaction from")
print("30 days ago counts half as much as one from today.")
print("\nRight: Affinity heatmap -- bright green = strong relationship, red = weak.")
print("Notice most pairs are weak (red) -- only a few pairs have strong affinity.")

## 2. Content Understanding Features

### Text Understanding (NLP)

Think of this like having a librarian who can read every post and summarize what it is about. We use **pre-trained language models** (like BERT) to convert text into a numerical vector (embedding) that captures meaning.

### Image/Video Understanding (Computer Vision)

Think of this like having an art critic who can look at every image and describe what is in it. We use **pre-trained vision models** (like ResNet or CLIP) to convert images into numerical vectors.

**Why embeddings?** Because models need numbers, not words or pictures. Embeddings convert human-readable content into a language that neural networks understand.

In [ ]:
# === Simulating Content Embeddings ===
# In production, these come from BERT (text) and ResNet/CLIP (images)
# Here we simulate them to demonstrate the concept

def simulate_text_embedding(text, embedding_dim=64):
    """
    Simulate a BERT-like text embedding.
    
    12-year-old version:
    Imagine BERT reads the post and converts it into a list of 64 numbers.
    Posts about similar topics will have similar numbers.
    'I love basketball' and 'Great NBA game' will have similar embeddings
    because BERT understands they are both about basketball.
    
    Staff-level detail:
    In production: Use BERT's [CLS] token embedding, or mean-pool the last
    hidden state. Fine-tune on downstream engagement prediction task.
    Key decision: freeze BERT vs fine-tune end-to-end (trade-off: compute vs quality).
    """
    # Simulate: hash the text to get a deterministic seed
    np.random.seed(hash(text) % 2**31)
    
    # Topic-based embedding: similar topics -> similar vectors
    topic_map = {
        'sports': np.array([1, 0, 0, 0, 0]),
        'food': np.array([0, 1, 0, 0, 0]),
        'travel': np.array([0, 0, 1, 0, 0]),
        'tech': np.array([0, 0, 0, 1, 0]),
        'music': np.array([0, 0, 0, 0, 1]),
    }
    
    # Detect topic from text (simple keyword matching for simulation)
    base = np.random.randn(embedding_dim) * 0.1
    for topic, vec in topic_map.items():
        if topic in text.lower():
            base[:5] += vec * 2.0
    
    # Normalize
    base = base / (np.linalg.norm(base) + 1e-8)
    return base


def simulate_image_embedding(image_type, embedding_dim=64):
    """
    Simulate a ResNet/CLIP-like image embedding.
    
    In production: pass image through ResNet-50 or CLIP, extract penultimate layer.
    """
    np.random.seed(hash(image_type) % 2**31)
    embedding = np.random.randn(embedding_dim) * 0.5
    # Add signal based on image type
    if image_type == 'selfie':
        embedding[0] += 2.0
    elif image_type == 'food_photo':
        embedding[1] += 2.0
    elif image_type == 'landscape':
        embedding[2] += 2.0
    elif image_type == 'meme':
        embedding[3] += 2.0
    return embedding / (np.linalg.norm(embedding) + 1e-8)


# Demonstrate similarity between related posts
posts = [
    "Amazing sports game last night!",
    "The sports team won the championship!",
    "I made the best food for dinner",
    "Check out this new tech gadget",
    "Traveling to sports events is fun",
]

embeddings = [simulate_text_embedding(post) for post in posts]

# Compute cosine similarity matrix
n = len(posts)
sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sim_matrix[i, j] = np.dot(embeddings[i], embeddings[j])

# Visualize
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(sim_matrix, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(n))
ax.set_yticks(range(n))
short_labels = [p[:25] + '...' if len(p) > 25 else p for p in posts]
ax.set_xticklabels(short_labels, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(short_labels, fontsize=9)
ax.set_title('Post Similarity (Cosine Similarity of Embeddings)', fontsize=13, fontweight='bold')

for i in range(n):
    for j in range(n):
        ax.text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center', fontsize=10)

plt.colorbar(im, label='Cosine Similarity')
plt.tight_layout()
plt.show()

print("Notice: Posts about 'sports' are more similar to each other than to 'food' or 'tech' posts.")
print("In production, BERT captures much richer semantic relationships.")

## 3. Social Graph Features

### The Power of Social Proof

Think about how you decide what to watch on Netflix. If your three best friends all said a movie was amazing, you would probably watch it too. That is **social proof** -- we trust the opinions of people we are close to.

Social graph features capture this by looking at:
- How many of your friends liked this post?
- Did your CLOSE friends engage with it?
- Is the author a friend of a friend (2-hop connection)?

In [ ]:
# === Social Graph Feature Engineering ===

# Build a friendship graph
np.random.seed(42)
n_users = 20

# Create friendship adjacency (symmetric)
friendships = set()
for _ in range(50):
    u1, u2 = np.random.randint(0, n_users, 2)
    if u1 != u2:
        friendships.add((min(u1, u2), max(u1, u2)))

# Close friends (subset of friendships)
close_friends = set()
for f in list(friendships)[:10]:
    close_friends.add(f)


def get_friends(user_id, friendships):
    """Get all friends of a user."""
    friends = set()
    for u1, u2 in friendships:
        if u1 == user_id:
            friends.add(u2)
        elif u2 == user_id:
            friends.add(u1)
    return friends


def compute_social_features(user_id, post_engagers, friendships, close_friends):
    """
    Compute social graph features for a (user, post) pair.
    
    12-year-old version:
    Look at who already liked/commented on this post.
    - How many of YOUR friends liked it?
    - Did your BEST friends like it?
    - How many total people engaged?
    
    Staff-level detail:
    - friend_engaged_count: number of user's friends who engaged
    - close_friend_engaged: binary flag -- any close friend engaged?
    - friend_engagement_ratio: friends_engaged / total_friends
    - social_proof_score: weighted count (close friends count more)
    - two_hop_engaged: friends-of-friends who engaged (weaker signal)
    """
    user_friends = get_friends(user_id, friendships)
    user_close_friends = set()
    for u1, u2 in close_friends:
        if u1 == user_id:
            user_close_friends.add(u2)
        elif u2 == user_id:
            user_close_friends.add(u1)
    
    # Friends who engaged with this post
    friends_engaged = user_friends & post_engagers
    close_friends_engaged = user_close_friends & post_engagers
    
    # Two-hop connections who engaged
    two_hop = set()
    for friend in user_friends:
        two_hop |= get_friends(friend, friendships)
    two_hop -= user_friends  # Remove direct friends
    two_hop.discard(user_id)  # Remove self
    two_hop_engaged = two_hop & post_engagers
    
    n_friends = max(len(user_friends), 1)
    
    return {
        'num_friends': len(user_friends),
        'friend_engaged_count': len(friends_engaged),
        'friend_engagement_ratio': len(friends_engaged) / n_friends,
        'close_friend_engaged': int(len(close_friends_engaged) > 0),
        'close_friend_engaged_count': len(close_friends_engaged),
        'two_hop_engaged_count': len(two_hop_engaged),
        'total_engagers': len(post_engagers),
        'social_proof_score': len(friends_engaged) * 1.0 + len(close_friends_engaged) * 3.0,
    }


# Demo: Post that several users engaged with
post_engagers = {2, 5, 7, 8, 12, 15, 18}

print("=== Social Graph Features ===")
print(f"Post engagers: {post_engagers}\n")

for user_id in [0, 3, 7, 10]:
    features = compute_social_features(user_id, post_engagers, friendships, close_friends)
    friends = get_friends(user_id, friendships)
    print(f"User {user_id} (friends: {friends}):")
    for k, v in features.items():
        print(f"  {k}: {v}")
    print()

In [ ]:
# === Visualizing Social Graph and Engagement ===

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Impact of friend engagement on user engagement
np.random.seed(42)
n_samples = 500
friend_engaged_counts = np.random.poisson(2, n_samples)
close_friend_flags = np.random.binomial(1, 0.2, n_samples)

# Simulate engagement probability (increases with friend engagement)
base_prob = 0.05
engagement_prob = base_prob + 0.08 * friend_engaged_counts + 0.15 * close_friend_flags
engagement_prob = np.clip(engagement_prob, 0, 1)
engaged = np.random.binomial(1, engagement_prob)

# Group by friend count and compute engagement rate
friend_counts = sorted(set(friend_engaged_counts))
engagement_rates = []
for fc in friend_counts:
    mask = friend_engaged_counts == fc
    if mask.sum() > 5:
        engagement_rates.append(engaged[mask].mean())
    else:
        engagement_rates.append(None)

valid_counts = [fc for fc, er in zip(friend_counts, engagement_rates) if er is not None]
valid_rates = [er for er in engagement_rates if er is not None]

axes[0].bar(valid_counts, valid_rates, color='#3498db', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Number of Friends Who Engaged', fontsize=12)
axes[0].set_ylabel('User Engagement Rate', fontsize=12)
axes[0].set_title('Social Proof Effect', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# Plot 2: Close friend vs regular friend impact
categories = ['No friend\nengaged', 'Regular friend\nengaged', 'Close friend\nengaged']
rates = [
    engaged[(friend_engaged_counts == 0) & (close_friend_flags == 0)].mean(),
    engaged[(friend_engaged_counts > 0) & (close_friend_flags == 0)].mean(),
    engaged[close_friend_flags == 1].mean(),
]

bar_colors = ['#95a5a6', '#3498db', '#2ecc71']
axes[1].bar(categories, rates, color=bar_colors, edgecolor='white', width=0.6)
axes[1].set_ylabel('User Engagement Rate', fontsize=12)
axes[1].set_title('Close Friends Matter More', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

for i, (cat, rate) in enumerate(zip(categories, rates)):
    axes[1].text(i, rate + 0.01, f'{rate:.1%}', ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("Key insight: The more friends who engaged with a post, the more likely YOU")
print("are to engage too. And close friends have an outsized effect.")

## 4. Embedding-Based Features

### User and Post Embeddings

Think of embeddings as a **fingerprint** -- a compact numerical representation that captures the essence of a user or a post. Users with similar interests will have similar fingerprints. Posts about similar topics will have similar fingerprints.

We learn these embeddings during training, so they automatically capture patterns the model finds useful.

In [ ]:
import torch
import torch.nn as nn

# === Embedding Layer Design ===

class NewsFeedEmbeddingLayer(nn.Module):
    """
    Embedding layer that converts categorical features into dense vectors.
    
    12-year-old version:
    Instead of saying 'User #42' (which means nothing to a model),
    we represent each user as a list of numbers that capture their
    personality. Users who behave similarly will have similar numbers.
    
    Staff-level detail:
    - User embeddings: capture latent user preferences
    - Author embeddings: capture latent author style
    - Topic embeddings: capture latent topic representations
    - Cross-feature embeddings: user-topic affinity via dot product
    """
    
    def __init__(self, n_users=1000, n_authors=500, n_topics=50,
                 user_embed_dim=32, author_embed_dim=32, topic_embed_dim=16):
        super().__init__()
        
        self.user_embedding = nn.Embedding(n_users, user_embed_dim)
        self.author_embedding = nn.Embedding(n_authors, author_embed_dim)
        self.topic_embedding = nn.Embedding(n_topics, topic_embed_dim)
        
        # Initialize with small random values
        nn.init.normal_(self.user_embedding.weight, std=0.01)
        nn.init.normal_(self.author_embedding.weight, std=0.01)
        nn.init.normal_(self.topic_embedding.weight, std=0.01)
    
    def forward(self, user_ids, author_ids, topic_ids):
        user_emb = self.user_embedding(user_ids)      # (batch, user_embed_dim)
        author_emb = self.author_embedding(author_ids)  # (batch, author_embed_dim)
        topic_emb = self.topic_embedding(topic_ids)    # (batch, topic_embed_dim)
        
        # Cross-feature: user-author compatibility (dot product)
        user_author_compat = (user_emb * author_emb).sum(dim=-1, keepdim=True)
        
        # Concatenate all embeddings + cross-features
        combined = torch.cat([
            user_emb,
            author_emb,
            topic_emb,
            user_author_compat,
        ], dim=-1)
        
        return combined


# Demo
embed_layer = NewsFeedEmbeddingLayer()

# Simulate a batch of 5 (user, author, topic) tuples
user_ids = torch.tensor([10, 20, 30, 10, 10])
author_ids = torch.tensor([5, 5, 5, 15, 25])
topic_ids = torch.tensor([3, 3, 7, 3, 3])

embeddings = embed_layer(user_ids, author_ids, topic_ids)

print("=== Embedding Layer ===")
print(f"User embedding dim: 32")
print(f"Author embedding dim: 32")
print(f"Topic embedding dim: 16")
print(f"User-author compatibility: 1 (dot product)")
print(f"Total embedding output dim: {embeddings.shape[1]}")
print(f"\nBatch output shape: {embeddings.shape}")

# Show how same user + different author gives different embeddings
print(f"\nSame user (10), different authors:")
print(f"  User 10 + Author 5:  first 5 dims = {embeddings[0, :5].detach().numpy().round(4)}")
print(f"  User 10 + Author 15: first 5 dims = {embeddings[3, :5].detach().numpy().round(4)}")
print(f"  User 10 + Author 25: first 5 dims = {embeddings[4, :5].detach().numpy().round(4)}")
print(f"\nThe user part stays the same, but the author part changes -- this captures")
print(f"different 'compatibility' between user 10 and each author.")

## 5. Multi-Task DNN Architecture: Full Implementation

Now let us put everything together into a complete ranking model that combines:
1. Embedding features (user, author, topic)
2. Dense features (affinity scores, social features, post features)
3. Multi-task heads (one per reaction type)

### Architecture Pattern: Shared-Bottom Multi-Task

```
Embeddings    Dense Features
    |              |
    v              v
  [Embed Layer] [BatchNorm]
    |              |
    +------+-------+
           |
           v
    [Shared Bottom Layers]  <-- learns cross-task patterns
    [256 -> 128 -> 64]
           |
    +--+--+--+--+--+
    |  |  |  |  |  |
    v  v  v  v  v  v
  [Click][Like][Comment][Share][Hide][Dwell]
  Head   Head  Head     Head   Head  Head
```

In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# === Complete Multi-Task Ranking Model ===

class FullNewsFeedRanker(nn.Module):
    """
    Complete multi-task ranking model for news feed personalization.
    
    12-year-old version:
    This model is like a team of experts who share a common brain.
    The 'shared brain' looks at all the features and figures out
    the big picture. Then each expert (head) specializes in predicting
    one specific thing (will they click? will they like? will they hide?).
    
    Staff-level detail:
    - Input: concatenation of embedding features + dense features
    - Shared bottom: 3-layer MLP with BatchNorm and Dropout
    - Task heads: 2-layer MLPs with task-specific activations
    - Binary tasks: sigmoid activation
    - Regression tasks: ReLU activation (non-negative dwell time)
    - Training: multi-task loss with task-specific weights
    """
    
    def __init__(self, n_dense_features, n_users=1000, n_authors=500, n_topics=50,
                 embed_dim=32, topic_embed_dim=16):
        super().__init__()
        
        # Embedding layers
        self.user_embedding = nn.Embedding(n_users, embed_dim)
        self.author_embedding = nn.Embedding(n_authors, embed_dim)
        self.topic_embedding = nn.Embedding(n_topics, topic_embed_dim)
        
        # Total input dim = embeddings + user-author compat + dense features
        total_embed_dim = embed_dim * 2 + topic_embed_dim + 1  # +1 for dot product
        total_input_dim = total_embed_dim + n_dense_features
        
        # Shared backbone
        self.shared = nn.Sequential(
            nn.Linear(total_input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
        )
        
        # Task-specific heads
        self.binary_tasks = ['click', 'like', 'comment', 'share', 'hide']
        self.regression_tasks = ['dwell_time']
        self.all_tasks = self.binary_tasks + self.regression_tasks
        
        self.task_heads = nn.ModuleDict()
        for task in self.all_tasks:
            self.task_heads[task] = nn.Sequential(
                nn.Linear(64, 32),
                nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(32, 1)
            )
    
    def forward(self, user_ids, author_ids, topic_ids, dense_features):
        # Get embeddings
        user_emb = self.user_embedding(user_ids)
        author_emb = self.author_embedding(author_ids)
        topic_emb = self.topic_embedding(topic_ids)
        user_author_compat = (user_emb * author_emb).sum(dim=-1, keepdim=True)
        
        # Concatenate all inputs
        x = torch.cat([user_emb, author_emb, topic_emb, user_author_compat, dense_features], dim=-1)
        
        # Shared backbone
        shared_repr = self.shared(x)
        
        # Task-specific outputs
        outputs = {}
        for task in self.binary_tasks:
            outputs[task] = torch.sigmoid(self.task_heads[task](shared_repr)).squeeze(-1)
        for task in self.regression_tasks:
            outputs[task] = torch.relu(self.task_heads[task](shared_repr)).squeeze(-1)
        
        return outputs


# Create model
n_dense_features = 12  # affinity scores + social features + post features
ranker = FullNewsFeedRanker(n_dense_features=n_dense_features)

print("=== Full News Feed Ranker ===")
print(f"Dense features: {n_dense_features}")
print(f"Binary tasks: {ranker.binary_tasks}")
print(f"Regression tasks: {ranker.regression_tasks}")

total_params = sum(p.numel() for p in ranker.parameters())
print(f"\nTotal parameters: {total_params:,}")

# Test forward pass
batch_size = 8
test_users = torch.randint(0, 1000, (batch_size,))
test_authors = torch.randint(0, 500, (batch_size,))
test_topics = torch.randint(0, 50, (batch_size,))
test_dense = torch.randn(batch_size, n_dense_features)

outputs = ranker(test_users, test_authors, test_topics, test_dense)
print(f"\nTest forward pass (batch_size={batch_size}):")
for task, pred in outputs.items():
    print(f"  {task}: shape={pred.shape}, range=[{pred.min().item():.4f}, {pred.max().item():.4f}]")

In [ ]:
# === Training the Full Ranker ===

def generate_full_training_data(n_samples=3000, n_dense=12, n_users=100, n_authors=50, n_topics=20):
    """
    Generate synthetic training data with user/author/topic IDs + dense features.
    """
    np.random.seed(42)
    
    user_ids = np.random.randint(0, n_users, n_samples)
    author_ids = np.random.randint(0, n_authors, n_samples)
    topic_ids = np.random.randint(0, n_topics, n_samples)
    dense_features = np.random.randn(n_samples, n_dense)
    
    # Create labels correlated with features
    # Higher affinity (dense_features[:, 0]) -> more engagement
    base_signal = 0.4 * dense_features[:, 0] + 0.3 * dense_features[:, 3] + 0.2 * dense_features[:, 6]
    
    labels = {}
    labels['click'] = (base_signal + np.random.randn(n_samples) * 0.5 > -0.2).astype(float)
    labels['like'] = (base_signal + np.random.randn(n_samples) * 0.5 > 0.3).astype(float)
    labels['comment'] = (base_signal + np.random.randn(n_samples) * 0.5 > 0.8).astype(float)
    labels['share'] = (base_signal + np.random.randn(n_samples) * 0.5 > 1.2).astype(float)
    labels['hide'] = ((-base_signal) + np.random.randn(n_samples) * 0.5 > 1.5).astype(float)
    labels['dwell_time'] = np.maximum(0, 3.0 + 2.0 * base_signal + np.random.randn(n_samples) * 2.0)
    
    return user_ids, author_ids, topic_ids, dense_features, labels


# Generate and split data
user_ids, author_ids, topic_ids, dense_features, labels = generate_full_training_data()
split = int(0.8 * len(user_ids))

# Training
from sklearn.metrics import roc_auc_score

optimizer = optim.Adam(ranker.parameters(), lr=0.001)
bce_fn = nn.BCELoss()
huber_fn = nn.HuberLoss()

task_weights = {'click': 1.0, 'like': 2.0, 'comment': 3.0, 'share': 4.0, 'hide': 2.0, 'dwell_time': 1.0}

# Prepare tensors
train_users = torch.LongTensor(user_ids[:split])
train_authors = torch.LongTensor(author_ids[:split])
train_topics = torch.LongTensor(topic_ids[:split])
train_dense = torch.FloatTensor(dense_features[:split])
train_labels = {k: torch.FloatTensor(v[:split]) for k, v in labels.items()}

test_users = torch.LongTensor(user_ids[split:])
test_authors = torch.LongTensor(author_ids[split:])
test_topics = torch.LongTensor(topic_ids[split:])
test_dense = torch.FloatTensor(dense_features[split:])
test_labels = {k: v[split:] for k, v in labels.items()}

# Create data loader
train_dataset = TensorDataset(
    train_users, train_authors, train_topics, train_dense,
    *[train_labels[t] for t in ranker.all_tasks]
)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

print("=== Training Full Ranker ===")
for epoch in range(25):
    ranker.train()
    epoch_loss = 0
    
    for batch in train_loader:
        b_users, b_authors, b_topics, b_dense = batch[0], batch[1], batch[2], batch[3]
        b_labels = {task: batch[4 + i] for i, task in enumerate(ranker.all_tasks)}
        
        optimizer.zero_grad()
        outputs = ranker(b_users, b_authors, b_topics, b_dense)
        
        loss = 0
        for task in ranker.binary_tasks:
            loss += task_weights[task] * bce_fn(outputs[task], b_labels[task])
        for task in ranker.regression_tasks:
            loss += task_weights[task] * huber_fn(outputs[task], b_labels[task])
        
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    if (epoch + 1) % 5 == 0:
        ranker.eval()
        with torch.no_grad():
            test_out = ranker(test_users, test_authors, test_topics, test_dense)
        aucs = {}
        for task in ranker.binary_tasks:
            try:
                aucs[task] = roc_auc_score(test_labels[task], test_out[task].numpy())
            except ValueError:
                aucs[task] = 0.5
        auc_str = " | ".join([f"{t}={v:.3f}" for t, v in aucs.items()])
        print(f"Epoch {epoch+1:2d} | Loss: {epoch_loss/len(train_loader):.4f} | {auc_str}")

## 6. Negative Feedback Handling

### Why Negative Signals Are Asymmetrically Important

Imagine a restaurant that serves you a bad meal. You might never go back, even if they served 100 great meals before. Negative experiences have an **outsized impact** on behavior.

In news feeds, the same is true:
- A user who **hides** a post is saying: "I do not want to see this kind of content"
- A user who **unfollows** someone is saying: "I am done with this person's posts entirely"
- A user who **reports** a post is saying: "This content should not exist on the platform"

These signals are **much stronger** than positive signals, and ignoring them leads to terrible user experience.

In [ ]:
# === Negative Feedback Analysis ===

# Simulate different types of negative feedback and their impact
np.random.seed(42)

negative_signals = {
    'hide': {
        'weight': -10,
        'scope': 'post',
        'action': 'Reduce score for similar posts from same author',
        'recovery': 'Gradual -- may show similar content again after weeks',
    },
    'unfollow': {
        'weight': -30,
        'scope': 'author',
        'action': 'Stop showing posts from this author entirely',
        'recovery': 'Only if user explicitly re-follows',
    },
    'report': {
        'weight': -50,
        'scope': 'post + author',
        'action': 'Flag post for review, reduce author visibility globally',
        'recovery': 'Requires content review team approval',
    },
    'block': {
        'weight': -100,
        'scope': 'author (permanent)',
        'action': 'Never show this author\'s content to this user again',
        'recovery': 'Only if user manually unblocks',
    },
}

# Visualize impact
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Weight comparison
positive_signals = {'click': 1, 'like': 5, 'comment': 10, 'share': 20}
all_signals = {**positive_signals, **{k: v['weight'] for k, v in negative_signals.items()}}

names = list(all_signals.keys())
weights = list(all_signals.values())
colors = ['#2ecc71' if w > 0 else '#e74c3c' for w in weights]

axes[0].barh(names, weights, color=colors, edgecolor='white')
axes[0].set_xlabel('Signal Weight', fontsize=12)
axes[0].set_title('Positive vs Negative Signal Weights', fontsize=14, fontweight='bold')
axes[0].axvline(x=0, color='black', linewidth=0.5)
for bar, w in zip(axes[0].patches, weights):
    x_pos = bar.get_width() + 2 if w > 0 else bar.get_width() - 8
    axes[0].text(x_pos, bar.get_y() + bar.get_height()/2, f'{w:+d}',
                 va='center', fontsize=10, fontweight='bold')

# Plot 2: How negative feedback propagates
days = np.arange(0, 60)
# After a 'hide', the author's score decays back gradually
hide_decay = 1.0 - 0.5 * np.exp(-days / 14)
# After an 'unfollow', the author's score stays at zero
unfollow_effect = np.where(days < 1, 1.0, 0.0)
# After a 'report', global effect
report_effect = np.where(days < 1, 1.0, 0.3 * np.exp(-days / 30))

axes[1].plot(days, hide_decay, 'orange', linewidth=2, label='After Hide (gradual recovery)')
axes[1].plot(days, unfollow_effect, 'red', linewidth=2, label='After Unfollow (permanent)')
axes[1].plot(days, report_effect, 'darkred', linewidth=2, label='After Report (global penalty)')
axes[1].axhline(y=1.0, color='green', linestyle='--', alpha=0.5, label='Normal score multiplier')
axes[1].set_xlabel('Days After Negative Feedback', fontsize=12)
axes[1].set_ylabel('Score Multiplier for Author', fontsize=12)
axes[1].set_title('Impact Duration of Negative Signals', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([-0.1, 1.2])

plt.tight_layout()
plt.show()

print("=== Negative Feedback Summary ===")
for signal, info in negative_signals.items():
    print(f"\n{signal.upper()}:")
    print(f"  Weight: {info['weight']}")
    print(f"  Scope: {info['scope']}")
    print(f"  Action: {info['action']}")
    print(f"  Recovery: {info['recovery']}")

In [ ]:
# === Integrating Negative Feedback into Ranking ===

def apply_negative_feedback_penalties(engagement_scores, user_negative_history, post_authors):
    """
    Apply penalties based on user's negative feedback history.
    
    12-year-old version:
    If you told us you don't like someone's posts before,
    we multiply their score by a small number to push them
    way down in your feed.
    
    Staff-level detail:
    - hide -> 0.3x multiplier for that author (decays over time)
    - unfollow -> 0.0x multiplier (complete suppression)
    - report -> 0.1x multiplier (near-complete suppression + global flag)
    - block -> 0.0x multiplier (hard filter, removed in retrieval stage)
    """
    penalty_multipliers = {
        'hide': 0.3,
        'unfollow': 0.0,
        'report': 0.1,
        'block': 0.0,
    }
    
    adjusted_scores = engagement_scores.copy()
    penalty_log = []
    
    for i, author in enumerate(post_authors):
        if author in user_negative_history:
            feedback_type = user_negative_history[author]
            multiplier = penalty_multipliers.get(feedback_type, 1.0)
            old_score = adjusted_scores[i]
            adjusted_scores[i] *= multiplier
            penalty_log.append({
                'post_idx': i,
                'author': author,
                'feedback': feedback_type,
                'old_score': old_score,
                'new_score': adjusted_scores[i],
            })
    
    return adjusted_scores, penalty_log


# Demo
np.random.seed(42)
n_posts = 15
raw_scores = np.random.uniform(1.0, 8.0, n_posts)
authors = [f'author_{np.random.randint(0, 8)}' for _ in range(n_posts)]

# User has negative history with some authors
negative_history = {
    'author_2': 'hide',
    'author_5': 'unfollow',
    'author_7': 'report',
}

adjusted_scores, penalties = apply_negative_feedback_penalties(raw_scores, negative_history, authors)

print("=== Negative Feedback Penalty Application ===")
print(f"User's negative history: {negative_history}\n")

# Show before/after ranking
original_rank = np.argsort(-raw_scores)
adjusted_rank = np.argsort(-adjusted_scores)

print(f"{'Rank':<6} {'Post':<8} {'Author':<12} {'Original':<10} {'Adjusted':<10} {'Penalty'}")
print("-" * 60)
for rank, idx in enumerate(adjusted_rank[:10]):
    penalty_str = ""
    for p in penalties:
        if p['post_idx'] == idx:
            penalty_str = f"({p['feedback']})"
    print(f"{rank+1:<6} Post_{idx:<4} {authors[idx]:<12} {raw_scores[idx]:<10.3f} {adjusted_scores[idx]:<10.3f} {penalty_str}")

print("\nNotice how posts from hidden/unfollowed/reported authors drop in ranking.")

## 7. Putting It All Together: Complete Ranking Pipeline

Let us combine everything into a single ranking pipeline that takes a user and a set of candidate posts, and returns a ranked feed.

In [ ]:
# === Complete Ranking Pipeline ===

class CompleteRankingPipeline:
    """
    Full ranking pipeline combining all components.
    
    Steps:
    1. Compute affinity features for each (user, post_author) pair
    2. Compute social graph features
    3. Compute content features
    4. Run multi-task model to get engagement predictions
    5. Compute weighted engagement score
    6. Apply negative feedback penalties
    7. Return ranked post list
    """
    
    def __init__(self, model, reaction_weights):
        self.model = model
        self.reaction_weights = reaction_weights
    
    def rank(self, user_id, candidate_posts, negative_history=None):
        """
        Rank candidate posts for a user.
        """
        n = len(candidate_posts)
        
        # Prepare model inputs (simulated for demo)
        user_ids = torch.LongTensor([user_id % 100] * n)
        author_ids = torch.LongTensor([p['author_id'] % 50 for p in candidate_posts])
        topic_ids = torch.LongTensor([p['topic_id'] % 20 for p in candidate_posts])
        dense_feats = torch.FloatTensor(np.random.randn(n, n_dense_features))
        
        # Get model predictions
        self.model.eval()
        with torch.no_grad():
            outputs = self.model(user_ids, author_ids, topic_ids, dense_feats)
        
        # Compute engagement scores
        scores = np.zeros(n)
        for task in self.model.binary_tasks:
            weight = self.reaction_weights.get(task, 0)
            scores += outputs[task].numpy() * weight
        
        # Apply negative feedback penalties
        if negative_history:
            post_authors = [f"author_{p['author_id']}" for p in candidate_posts]
            scores, _ = apply_negative_feedback_penalties(scores, negative_history, post_authors)
        
        # Rank
        ranked_indices = np.argsort(-scores)
        
        results = []
        for rank, idx in enumerate(ranked_indices):
            results.append({
                'rank': rank + 1,
                'post': candidate_posts[idx],
                'score': scores[idx],
                'predictions': {t: outputs[t][idx].item() for t in self.model.binary_tasks},
            })
        
        return results


# Create pipeline
reaction_weights = {'click': 1, 'like': 5, 'comment': 10, 'share': 20, 'hide': -10}
pipeline = CompleteRankingPipeline(ranker, reaction_weights)

# Generate candidate posts
np.random.seed(42)
candidate_posts = []
topics = ['sports', 'food', 'travel', 'tech', 'music', 'pets', 'politics']
for i in range(25):
    candidate_posts.append({
        'post_id': i,
        'author_id': np.random.randint(0, 20),
        'topic_id': np.random.randint(0, len(topics)),
        'topic': topics[np.random.randint(0, len(topics))],
        'text': f'Post about {topics[np.random.randint(0, len(topics))]} #{i}',
    })

# Rank for user 42
neg_history = {'author_5': 'hide', 'author_12': 'unfollow'}
ranked_feed = pipeline.rank(user_id=42, candidate_posts=candidate_posts, negative_history=neg_history)

print("=== Ranked News Feed for User 42 ===")
print(f"Negative history: {neg_history}\n")
print(f"{'Rank':<6} {'Post':<25} {'Author':<10} {'Score':<8} {'P(like)':<9} {'P(share)':<9} {'P(hide)'}")
print("-" * 75)
for r in ranked_feed[:12]:
    p = r['post']
    preds = r['predictions']
    print(f"{r['rank']:<6} {p['text'][:23]:<25} auth_{p['author_id']:<5} {r['score']:<8.3f} "
          f"{preds['like']:<9.3f} {preds['share']:<9.3f} {preds['hide']:.3f}")

## Key Takeaways

1. **User-author affinity is the most important feature** -- compute it from interaction history with time decay to emphasize recent behavior

2. **Content understanding** via NLP (BERT) and CV (ResNet/CLIP) turns unstructured posts into numerical features the model can use

3. **Social graph features** capture social proof -- if your close friends engaged with a post, you are much more likely to engage too

4. **Embedding layers** learn latent representations of users, authors, and topics that capture compatibility through dot-product interactions

5. **Multi-task architecture** with shared backbone + task heads is more efficient than separate models and helps sparse tasks via transfer learning

6. **Negative feedback is asymmetrically important** -- a single hide or block carries more weight than several likes, and must be handled carefully with appropriate multipliers and recovery policies

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)